## 2- Data Preprocessing

In [1]:
import pandas as pd
import numpy as np

In [2]:
#loading train dataset
data = pd.read_csv('ds/fraudTrain.csv' )

In [3]:
data.head(1)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0


In [4]:
#remove unecesary Columns 
data.drop(columns=['Unnamed: 0','cc_num','first', 'last', 'street' ,
                                  'trans_num'],inplace=True)

In [5]:
data.columns

Index(['trans_date_trans_time', 'merchant', 'category', 'amt', 'gender',
       'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob',
       'unix_time', 'merch_lat', 'merch_long', 'is_fraud'],
      dtype='str')

In [6]:
data.nunique()

#deleting also unix_time no use  , we can  extract time from others
#removing also city , state , zip too much unique values and also it could result
# in many columns or variables while encoding (and already covered in long and lat)
data.drop(columns=['city','zip','state', 'merchant' , 'unix_time'],inplace=True)

#while we keep city population if it s a small town with hight amount this could be a fraud
#and also merch_lat and long , the distance between where transaction made and where the actuale
#merchant location

In [7]:
data.head(1)

,trans_date_trans_time,category,amt,gender,lat,long,city_pop,job,dob,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,misc_net,4.97,F,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,36.011293,-82.048315,0


In [8]:
#convert to date time from string

data["trans_date_trans_time"] = pd.to_datetime(data['trans_date_trans_time'])

data['dob'] = pd.to_datetime(data['dob'])

#extract

#trans hour
data['trans_hour'] = data['trans_date_trans_time'].dt.hour
#trans month
data['trans_month'] = data['trans_date_trans_time'].dt.month
#trans day
data['trans_day'] = data['trans_date_trans_time'].dt.day
#trans year
data['trans_year'] = data['trans_date_trans_time'].dt.year

In [9]:
#age based of date of birth and also based on date of transaction
data['age'] = (
    (data['trans_date_trans_time'] - data['dob']) / pd.Timedelta(days=365.25)
).astype(int)

In [10]:
#now after extracting data we can delete the variables

data.drop(columns=["trans_date_trans_time" , "dob"] ,inplace=True)

In [11]:
data.head(4)

,category,amt,gender,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,trans_hour,trans_month,trans_day,trans_year,age
0,misc_net,4.97,F,36.0788,-81.1781,3495,"Psychologist, counselling",36.011293,-82.048315,0,0,1,1,2019,30
1,grocery_pos,107.23,F,48.8878,-118.2105,149,Special educational needs teacher,49.159047,-118.186462,0,0,1,1,2019,40
2,entertainment,220.11,M,42.1808,-112.2620,4154,Nature conservation officer,43.150704,-112.154481,0,0,1,1,2019,56
3,gas_transport,45.00,M,46.2306,-112.1138,1939,Patent attorney,47.034331,-112.561071,0,0,1,1,2019,51


In [12]:
from sklearn.preprocessing import LabelEncoder

#encode categorical data job category gender
#fix "encoder for each variables required for pkl files implimentation"
cat_encoder = LabelEncoder()
data['category'] = cat_encoder.fit_transform(data['category'])
gender_encoder = LabelEncoder()
data['gender'] = gender_encoder.fit_transform(data['gender'])
job_encoder = LabelEncoder()
data['job'] = job_encoder.fit_transform(data['job'])

In [13]:
data.head(3)

,category,amt,gender,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,trans_hour,trans_month,trans_day,trans_year,age
0,8,4.97,0,36.0788,-81.1781,3495,370,36.011293,-82.048315,0,0,1,1,2019,30
1,4,107.23,0,48.8878,-118.2105,149,428,49.159047,-118.186462,0,0,1,1,2019,40
2,0,220.11,1,42.1808,-112.2620,4154,307,43.150704,-112.154481,0,0,1,1,2019,56


In [14]:
print(data.isnull().sum())
#no null values

print(data.duplicated())
#no duplicates

category       0
amt            0
gender         0
lat            0
long           0
city_pop       0
job            0
merch_lat      0
merch_long     0
is_fraud       0
trans_hour     0
trans_month    0
trans_day      0
trans_year     0
age            0
dtype: int64
0          False
1          False
2          False
3          False
4          False
           ...  
1296670    False
1296671    False
1296672    False
1296673    False
1296674    False
Length: 1296675, dtype: bool


In [15]:
from sklearn.preprocessing import StandardScaler

#scall data

scaler = StandardScaler()
# Scale numerical data 
to_scale = ['amt', 'lat', 'long', 'city_pop' , 'merch_lat', 'merch_long' , 'trans_year'
            , 'trans_hour' , 'trans_day' , 'trans_month' , 'age']
data[to_scale] = scaler.fit_transform(data[to_scale])

In [16]:
data.head(1)

,category,amt,gender,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,trans_hour,trans_month,trans_day,trans_year,age
0,8,-0.407826,0,-0.48442,0.65762,-0.282589,370,-0.494354,0.593864,0,-1.878145,-1.504564,-1.652258,-0.634065,-0.890761


In [17]:
#save dataset to new file
data.to_csv('ds/Fraud_cleaned_dataset.csv')

In [18]:
#save scaling objects used in model implementation

import joblib

#encoders each one needs a file
joblib.dump(cat_encoder, 'categorie_encoder.pkl')
joblib.dump(job_encoder, 'job_encoder.pkl')
joblib.dump(gender_encoder, 'gender_encoder.pkl')

#scaller ok to be in one file
joblib.dump(scaler, 'scaler.pkl')


['scaler.pkl']